# SARR ETL — PyPI BigQuery → Embed (GPU) → Qdrant Cloud

Reads **current PyPI metadata** from `bigquery-public-data.pypi.distribution_metadata`
(one row per project, latest release). Unfiltered full corpus is **~1M** rows — too large
for **Qdrant free tier** (4 GB disk). Use the sizing env vars below (~150k–170k).

Libraries.io is joined for GitHub stars / SourceRank (ranking signals).

**Colab checklist**
1. Runtime → **GPU** (T4 or better)
2. Set `GCP_PROJECT_ID` to **your** GCP project (billing/quota only)
3. Set `QDRANT_URL`, `QDRANT_API_KEY`, `QDRANT_COLLECTION`
4. First full load: `LAST_UPDATE_DATE = "1970-01-01"` and delete `data/last_update_date.txt`
5. Incremental runs: keep `data/last_update_date.txt` (checkpointed after each Qdrant batch)

In [ ]:
# Clone or mount the repo, then install ETL extras.

import os
import sys
from pathlib import Path

# --- Option A (recommended): clone from GitHub ---
REPO_URL = "https://github.com/kanchana123/sarr-recommendation-api.git"
BRANCH = "feature/pypi-full-etl"  # switch to main after merge
project_root = Path("/content/sarr-recommendation-api")
if not project_root.is_dir():
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {project_root}
else:
    !git -C {project_root} fetch origin {BRANCH}
    !git -C {project_root} checkout {BRANCH}
    !git -C {project_root} pull --ff-only origin {BRANCH}

%cd {project_root}
!pip install -q -e ".[etl]"
print("Project root:", project_root)

In [ ]:
# --- Option B: Google Drive (uncomment if you sync the repo there) ---
# from google.colab import drive
# drive.mount('/content/drive')
# project_root = Path('/content/drive/MyDrive/sarr-recommendation-api')
# %cd {project_root}
# !pip install -q -e ".[etl]"

In [ ]:
import os
from pathlib import Path

# ========== EDIT THESE with REAL values ==========
os.environ["GCP_PROJECT_ID"] = "your-real-gcp-project-id"  # NOT bigquery-public-data
os.environ["ETL_EXTRACT_SOURCE"] = "pypi"
os.environ["BQ_PYPI_PROJECT"] = "bigquery-public-data"
os.environ["BQ_PYPI_DATASET"] = "pypi"
os.environ["BQ_LIBRARIES_PROJECT"] = "bigquery-public-data"
os.environ["BQ_LIBRARIES_DATASET"] = "libraries_io"

# Corpus filters (~500k–700k). Vectors-only in Qdrant; metadata at search time from BQ.
os.environ["QDRANT_VECTORS_ONLY"] = "true"
os.environ["ETL_MIN_DESCRIPTION_LENGTH"] = "25"
os.environ["ETL_MIN_LONG_DESCRIPTION_LENGTH"] = "80"
os.environ["ETL_ACTIVE_WITHIN_DAYS"] = "2920"
os.environ["ETL_REQUIRE_ANY_POPULARITY"] = "true"
os.environ["ETL_MIN_STARS"] = "1"
os.environ["ETL_MIN_FORKS"] = "1"
os.environ["ETL_MIN_SOURCE_RANK"] = "4"
os.environ["ETL_MIN_DEPENDENT_PROJECTS"] = "1"
os.environ["ETL_MAX_PACKAGES"] = "700000"

os.environ["QDRANT_URL"] = "https://xxxx.aws.cloud.qdrant.io"
os.environ["QDRANT_API_KEY"] = "your-real-qdrant-api-key"
os.environ["QDRANT_COLLECTION"] = "sarr"

os.environ["EMBEDDING_MODEL"] = "BAAI/bge-small-en-v1.5"
os.environ["EMBEDDING_DIM"] = "384"
os.environ["LAST_UPDATE_DATE"] = "1970-01-01"
# ================================================

assert "your-real" not in os.environ["GCP_PROJECT_ID"], "Set a real GCP_PROJECT_ID"
assert "xxxx" not in os.environ["QDRANT_URL"], "Set a real QDRANT_URL"
assert os.environ["QDRANT_API_KEY"] != "your-real-qdrant-api-key", "Set a real QDRANT_API_KEY"

from google.colab import auth

auth.authenticate_user()

Path("data").mkdir(exist_ok=True)
# Full reload: delete the checkpoint file before running ETL
# Path("data/last_update_date.txt").unlink(missing_ok=True)

print("Auth OK")
print("Billing project:", os.environ["GCP_PROJECT_ID"])
print("Extract source:", os.environ["ETL_EXTRACT_SOURCE"])
print("Qdrant collection:", os.environ["QDRANT_COLLECTION"])

In [ ]:
# Diagnostic: expect hundreds of thousands of rows on a full load (1970 watermark).
from sarr.common.config import get_settings
from sarr.etl.extract import count_extract_rows
from sarr.etl.validate import validate_etl_settings
from sarr.etl.watermark import load_watermark

get_settings.cache_clear()
settings = get_settings()

problems = validate_etl_settings(settings)
if problems:
    raise ValueError("Fix config first:\n- " + "\n- ".join(problems))

watermark = load_watermark("data/last_update_date.txt", default=settings.last_update_date)
print("Running count query (may take 1–3 min on first full load)...")
n = count_extract_rows(last_update_date=watermark, settings=settings)
print("Watermark:", watermark)
print("Rows to process:", n)
if n == 0:
    raise RuntimeError(
        "BigQuery returned 0 rows. Delete data/last_update_date.txt, set "
        "LAST_UPDATE_DATE=1970-01-01, or check ETL_EXTRACT_SOURCE=pypi."
    )
if watermark == "1970-01-01" and n < 100_000:
    print(
        "WARNING: full-load count is lower than expected (~500k+). "
        "Confirm ETL_EXTRACT_SOURCE=pypi and query completed."
    )

In [ ]:
import torch
from pathlib import Path
from sarr.common.config import get_settings
from sarr.etl.embed import BatchEmbedder
from sarr.etl.pipeline import run_etl
from sarr.etl.watermark import load_watermark

assert torch.cuda.is_available(), "Enable a GPU runtime before running ETL"

get_settings.cache_clear()
settings = get_settings()
watermark_path = "data/last_update_date.txt"
Path("data").mkdir(exist_ok=True)

resume_from = load_watermark(watermark_path, default=settings.last_update_date)
print("Resuming from watermark:", resume_from)
print("Qdrant:", settings.qdrant_url, "collection=", settings.qdrant_collection)
print("Extract:", settings.etl_extract_source)

embedder = BatchEmbedder(settings.embedding_model, device="cuda")

# Full corpus: 128–256 batch size on T4; reduce if you hit CUDA OOM.
stats = run_etl(
    last_update_date=None,
    batch_size=128,
    watermark_path=watermark_path,
    settings=settings,
    embedder=embedder,
)
print(stats)
if stats["processed"] == 0:
    raise RuntimeError("Nothing uploaded. Re-run the diagnostic cell above.")
print("OK — verify point count in Qdrant Cloud UI")
print("Download data/last_update_date.txt before closing Colab (resume checkpoint)")